# FAseg — Fine-tuning Metrics: Ex-Vivo & In-Vivo

Compares model performance **before vs after fine-tuning** on both datasets.

## Ex-Vivo (read-only)
Reads pre-computed inference masks and reports FA-boundary + pixel metrics
for the **Full** pretraining model across 3 states (UNSEEN sources only).

## In-Vivo (inference + metrics)
Part 1 runs the **no-fine-tuning** model and saves the result — comment these
cells out after the first run (the result is already saved).  Part 2 loads the
saved no-FT and fine-tuned masks and reports the same metrics.

**Manual labels (in-vivo)**: `masks/syf_leg_new` (displayed as "in vivo").

In [1]:
import os, sys, json, time
import numpy as np
import torch
from scipy.ndimage import zoom
from torch.utils.data import DataLoader, TensorDataset

_NB_DIR = os.path.abspath('')
if os.path.basename(_NB_DIR) == 'scripts':
    _project_root = os.path.dirname(_NB_DIR)
else:
    _project_root = _NB_DIR
_src_root = os.path.join(_project_root, 'src')
if _src_root not in sys.path:
    sys.path.insert(0, _src_root)

from faseg.models import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COLUMN_DT = 2.4e-7  # seconds per column (shared by ex-vivo and in-vivo metrics)

print(f'Project root : {_project_root}')
print(f'Device       : {device}')

Project root : /data/projects/AgentWork/FAseg for github
Device       : cuda


# =========================================================================
# Ex-Vivo — Full Pretraining: 3-State Comparison (read-only)
# =========================================================================
Reads already-saved inference masks; no model is loaded here.

In [2]:
# =========================================================================
# Ex-vivo paths
# =========================================================================
MANUAL_DIR = os.path.join(_project_root, 'manual segmentation', 'manual_seg_beef')
INFER_DIR  = os.path.join(_project_root, 'outputs', 'inference_results', 'exvivo')

# 3 states to compare (key -> label)
STATES = [
    ('without_fine_tuning/pretraining_best_s3', 'No Fine-tuning (best_s3)'),
    ('pretraining_best_s3',                     'best_s3 Fine-tuned'),
    ('pretraining_latest',                      'latest Fine-tuned'),
]

# UNSEEN split (matches fine-tuning training range)
TRAIN_MIN_SRC = 128
TRAIN_MAX_SRC = 384

print(f'Manual masks : {MANUAL_DIR}')
print(f'Inference    : {INFER_DIR}')
print(f'UNSEEN: src 1–{TRAIN_MIN_SRC-1} + {TRAIN_MAX_SRC+1}–511')

Manual masks : /data/projects/AgentWork/FAseg for github/manual segmentation/manual_seg_beef
Inference    : /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo
UNSEEN: src 1–127 + 385–511


In [3]:
def load_exvivo_mask(mask_path):
    """Load .npy mask -> (384, 384) bool, following RealLimbDataset2 pipeline."""
    mask = np.load(mask_path)                     # (512, 620) or (620, 512)
    mask = np.transpose(mask)                     # -> (620, 512) or (512, 620)
    target_height = 4955 // 6                     # 826
    scale_factor = target_height / mask.shape[1]
    mask_resized = zoom(mask, (1, scale_factor), order=0)  # nearest-neighbour
    mask = mask_resized[0:384, 300:684]           # (384, 384)
    return mask.astype(bool)


exvivo_masks = {}
for f in sorted(os.listdir(MANUAL_DIR)):
    if f.endswith('_mask.npy'):
        src = int(f.replace('src', '').replace('_mask.npy', ''))
        exvivo_masks[src] = load_exvivo_mask(os.path.join(MANUAL_DIR, f))

print(f'Loaded {len(exvivo_masks)} manual masks')
print(f'  Shape: {exvivo_masks[list(exvivo_masks.keys())[0]].shape}')

Loaded 255 manual masks
  Shape: (384, 384)


In [5]:
infer_results = {}
for rel_path, label in STATES:
    path = os.path.join(INFER_DIR, rel_path, 'pred_mask_3d.npy')
    if os.path.exists(path):
        infer_results[label] = np.load(path).astype(bool)
        print(f'{label:35s}  {infer_results[label].shape}  ✅')

if len(infer_results) == 0:
    raise RuntimeError('No inference results found!')

No Fine-tuning (best_s3)             (384, 384, 512)  ✅
best_s3 Fine-tuned                   (384, 384, 512)  ✅


In [6]:
def tof_boundary(mask_2d):
    """First '1' column per row; NaN if no foreground."""
    H = mask_2d.shape[0]
    cols = np.full(H, np.nan)
    for r in range(H):
        ones = np.where(mask_2d[r, :])[0]
        if len(ones) > 0:
            cols[r] = float(ones[0])
    return cols


def compute_metrics(manual_2d, pred_2d, column_dt):
    """TOF boundary errors + pixel precision/recall/F1 for one (manual, pred) pair."""
    m_bdry = tof_boundary(manual_2d)
    p_bdry = tof_boundary(pred_2d)
    valid = ~np.isnan(m_bdry) & ~np.isnan(p_bdry)
    if valid.sum() == 0:
        return {'max_err_s': np.nan, 'mae_s': np.nan, 'mean_err_s': np.nan,
                'var_s2': np.nan, 'precision': np.nan, 'recall': np.nan,
                'f1': np.nan, 'n_rows': 0}
    errors = (p_bdry[valid] - m_bdry[valid]) * column_dt
    inter = (manual_2d & pred_2d).sum()
    precision = inter / pred_2d.sum()  if pred_2d.sum() > 0  else 0.0
    recall    = inter / manual_2d.sum() if manual_2d.sum() > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'max_err_s': np.abs(errors).max(), 'mae_s': np.abs(errors).mean(),
            'mean_err_s': errors.mean(), 'var_s2': errors.var(),
            'precision': precision, 'recall': recall, 'f1': f1,
            'n_rows': int(valid.sum())}


def filter_unseen(per_src):
    """Keep only UNSEEN sources (1–127 + 385–511)."""
    return [s for s in per_src if s['src'] < TRAIN_MIN_SRC or s['src'] > TRAIN_MAX_SRC]


def macro_average(per_src):
    """Macro-average across sources, skipping NaN."""
    keys = ['max_err_s', 'mae_s', 'mean_err_s', 'var_s2', 'precision', 'recall', 'f1']
    agg = {}
    for k in keys:
        vals = np.array([s[k] for s in per_src if not np.isnan(s[k])])
        agg[k] = vals.mean() if len(vals) > 0 else np.nan
    agg['n_src'] = len(per_src)
    return agg

In [7]:
ALL_METRICS = {}  # label -> list of per-source dicts
for label, infer_3d in infer_results.items():
    per_src = []
    for src in sorted(exvivo_masks.keys()):
        if (src - 1) >= infer_3d.shape[2]:
            continue
        m = compute_metrics(exvivo_masks[src], infer_3d[:, :, src - 1], COLUMN_DT)
        m['src'] = src
        per_src.append(m)
    ALL_METRICS[label] = per_src
    print(f'{label}: {len(per_src)} sources computed')

No Fine-tuning (best_s3): 255 sources computed
best_s3 Fine-tuned: 255 sources computed


In [8]:
group_label = f'UNSEEN (1–{TRAIN_MIN_SRC-1} + {TRAIN_MAX_SRC+1}–511)'

print(f'\n{"="*140}')
print(f'  Ex-Vivo — Full Pretraining Model — 3-State Comparison  |  {group_label}')
print(f'  Errors: prediction − manual.  Negative bias → model predicts earlier.')
print(f'{"="*140}')
header = (f'{"State":35s}  {"Max Err":>10s}  {"MAE":>10s}  '
          f'{"Bias":>10s}  {"Var (s²)":>10s}  '
          f'{"Precision":>10s}  {"Recall":>10s}  {"F1(%)":>10s}  {"n_src":>6s}')
print(header)
print('-' * 140)

for label in [l for _, l in STATES]:
    if label not in ALL_METRICS:
        continue
    agg = macro_average(filter_unseen(ALL_METRICS[label]))
    print(f'{label:35s}  {agg["max_err_s"]:10.2e}  {agg["mae_s"]:10.2e}  '
          f'{agg["mean_err_s"]:10.2e}  {agg["var_s2"]:10.2e}  '
          f'{agg["precision"]:10.4f}  {agg["recall"]:10.4f}  {agg["f1"]*100:10.2f}  {agg["n_src"]:6d}')


  Ex-Vivo — Full Pretraining Model — 3-State Comparison  |  UNSEEN (1–127 + 385–511)
  Errors: prediction − manual.  Negative bias → model predicts earlier.
State                                   Max Err         MAE        Bias    Var (s²)   Precision      Recall       F1(%)   n_src
--------------------------------------------------------------------------------------------------------------------------------------------
No Fine-tuning (best_s3)               4.48e-06    1.43e-06   -1.12e-06    1.79e-12      0.9745      0.9942       98.42     128
best_s3 Fine-tuned                     2.51e-06    4.15e-07   -5.98e-08    1.16e-12      0.9939      0.9950       99.45     128


# =========================================================================
# In-Vivo — (Part 1) No-FT Inference & Save
# =========================================================================
> **Comment out cells 11–14 after the first run** — the no-FT result is already
> saved to `outputs/without_fine_tuning/`.

In [ ]:
# # =========================================================================
# # In-vivo inference paths
# # =========================================================================
# BIN_PATH     = os.path.join(_project_root, 'manual segmentation', 'syf_upper_leg')
# PRETRAIN_DIR = os.path.join(_project_root, 'outputs', 'pretraining')      # Full pretraining
# CKPT_PATH    = os.path.join(PRETRAIN_DIR, 'unet_epoch84.pth')            # best_s3 checkpoint
# CONFIG_PATH  = os.path.join(PRETRAIN_DIR, 'config.txt')
# SAVE_DIR     = os.path.join(_project_root, 'outputs', 'without_fine_tuning')
# os.makedirs(SAVE_DIR, exist_ok=True)

# print(f'Checkpoint : {CKPT_PATH}')
# print(f'Output     : {SAVE_DIR}')

Checkpoint : /data/projects/AgentWork/FAseg for github/outputs/pretraining/unet_epoch84.pth
Output     : /data/projects/AgentWork/FAseg for github/outputs/without_fine_tuning


In [ ]:
# def load_and_preprocess(bin_path):
#     raw = np.fromfile(bin_path, dtype=np.float32)
#     S = raw.size // (512 * 876)
#     data = raw.reshape((S, 512, 876))
#     for i in range(S):
#         shift = i if S == 512 else (2 * i)
#         data[i] = np.concatenate([data[i, shift:], data[i, :shift]], axis=0)
#     sub = data[:, 64:448, 300:684].astype(np.float32)
#     for i in range(S):
#         vmax = np.abs(sub[i]).max()
#         if vmax > 0:
#             sub[i] /= vmax
#     sub = np.clip(sub, -1.0, 1.0)
#     return sub

# print('Loading & preprocessing in-vivo data ...')
# stack = load_and_preprocess(BIN_PATH)
# S, H, W = stack.shape
# print(f'Preprocessed: {stack.shape}')

# tensor_stack = torch.from_numpy(stack).unsqueeze(1)
# dataset = TensorDataset(tensor_stack)
# loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

Loading & preprocessing in-vivo data ...
Preprocessed: (512, 384, 384)


In [ ]:
# if os.path.exists(CONFIG_PATH):
#     with open(CONFIG_PATH) as f:
#         cfg = json.load(f)
#     base_ch = cfg.get('base_channel', 64)
#     dropout = cfg.get('dropout_prob', 0.2)
#     use_bn  = cfg.get('use_bn', True)
# else:
#     base_ch, dropout, use_bn = 64, 0.2, True

# model = UNet(in_ch=1, base_ch=base_ch, num_classes=2,
#              dropout_prob=dropout, use_bn=use_bn).to(device)

# raw = torch.load(CKPT_PATH, map_location=device, weights_only=False)
# if isinstance(raw, dict) and 'model_state_dict' in raw:
#     state = raw['model_state_dict']
# else:
#     state = raw
# model.load_state_dict(state)
# model.eval()
# print(f'Loaded pretrained model (epoch {raw.get("epoch", "?") if isinstance(raw, dict) else "?"})')

# all_masks = []
# t0 = time.time()
# with torch.no_grad():
#     for (batch,) in loader:
#         batch = batch.to(device)
#         logits = model(batch)
#         preds = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
#         all_masks.append(preds)
# pred_masks = np.concatenate(all_masks, axis=0)
# elapsed = time.time() - t0
# print(f'Inference done: {S} slices in {elapsed:.1f}s  ({S/elapsed:.0f} slices/s)')

Loaded pretrained model (epoch ?)


/home/yifei-sun/anaconda3/envs/faseg_ablation/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Inference done: 512 slices in 3.0s  (173 slices/s)


In [ ]:
# boundary_matrix = np.full((H, S), np.nan, dtype=np.float32)
# for s in range(S):
#     for row in range(H):
#         ones = np.where(pred_masks[s, row, :] == 1)[0]
#         if len(ones) > 0:
#             boundary_matrix[row, s] = float(ones[0])

# mask_3d = pred_masks.transpose(1, 2, 0)
# np.save(os.path.join(SAVE_DIR, 'pred_mask_3d.npy'), mask_3d)
# np.asfortranarray(mask_3d).tofile(os.path.join(SAVE_DIR, 'pred_mask_3d.bin'))
# np.save(os.path.join(SAVE_DIR, 'boundary_matrix.npy'), boundary_matrix)
# with open(os.path.join(SAVE_DIR, 'inference_time.txt'), 'w') as f:
#     f.write(f'{elapsed:.2f}s  ({S} slices, {S/elapsed:.0f} slices/s)\n')
# print(f'Results saved to {SAVE_DIR}')

Results saved to /data/projects/AgentWork/FAseg for github/outputs/without_fine_tuning


# =========================================================================
# In-Vivo — (Part 2) Metrics (read from saved results)
# =========================================================================

In [19]:
# =========================================================================
# In-vivo metrics paths (self-contained: independent of Part 1)
# =========================================================================
# Manual label subdir under 'manual segmentation/masks/'.
INVIVO_MANUAL_SUBDIR = 'syf_leg'

INVIVO_MANUAL_DIR = os.path.join(_project_root, 'manual segmentation', 'masks', INVIVO_MANUAL_SUBDIR)
NOFT_RESULT = os.path.join(_project_root, 'outputs', 'without_fine_tuning', 'pred_mask_3d.npy')
FT_RESULT   = os.path.join(_project_root, 'outputs', 'inference_results', 'invivo', 'pretraining_best_s3', 'pred_mask_3d.npy')

print(f'Manual labels : {INVIVO_MANUAL_DIR}')
print(f'No-FT result  : {NOFT_RESULT}')
print(f'Fine-tuned    : {FT_RESULT}')

Manual labels : /data/projects/AgentWork/FAseg for github/manual segmentation/masks/syf_leg
No-FT result  : /data/projects/AgentWork/FAseg for github/outputs/without_fine_tuning/pred_mask_3d.npy
Fine-tuned    : /data/projects/AgentWork/FAseg for github/outputs/inference_results/invivo/pretraining_best_s3/pred_mask_3d.npy


In [20]:
# Manual masks (already (384, 384); load directly)
manual_masks = {}
for f in sorted(os.listdir(INVIVO_MANUAL_DIR)):
    if f.endswith('.npy'):
        src = int(f.replace('src', '').replace('.npy', ''))
        manual_masks[src] = np.load(os.path.join(INVIVO_MANUAL_DIR, f)).astype(bool)

noft_infer = np.load(NOFT_RESULT).astype(bool)
ft_infer   = np.load(FT_RESULT).astype(bool)

print(f'Manual masks : {len(manual_masks)} sources')
print(f'No-FT        : {noft_infer.shape}')
print(f'Fine-tuned   : {ft_infer.shape}')

Manual masks : 256 sources
No-FT        : (384, 384, 512)
Fine-tuned   : (384, 384, 512)


In [21]:
def compare_one(manual_dict, infer_3d):
    per_src = []
    for src in sorted(manual_dict.keys()):
        if (src - 1) >= infer_3d.shape[2]:
            continue
        m = compute_metrics(manual_dict[src], infer_3d[:, :, src - 1], COLUMN_DT)
        m['src'] = src
        per_src.append(m)
    return macro_average(per_src), len(per_src)

agg_noft, n_noft = compare_one(manual_masks, noft_infer)
agg_ft,   n_ft   = compare_one(manual_masks, ft_infer)

print(f'\n{"="*110}')
print(f'  in vivo  —  No Fine-tuning  vs  Fine-tuned (best_s3)  ({n_noft} sources)')
print(f'  Errors: prediction - manual')
print(f'{"="*110}')
header = (f'{"Model":20s}  {"Max Err":>10s}  {"MAE":>10s}  {"Bias":>10s}  '
          f'{"Var (s^2)":>10s}  {"Precision":>10s}  {"Recall":>10s}  {"F1(%)":>10s}')
print(header)
print('-' * 110)
print(f'{"No fine-tuning":20s}  {agg_noft["max_err_s"]:10.2e}  {agg_noft["mae_s"]:10.2e}  '
      f'{agg_noft["mean_err_s"]:10.2e}  {agg_noft["var_s2"]:10.2e}  '
      f'{agg_noft["precision"]:10.4f}  {agg_noft["recall"]:10.4f}  {agg_noft["f1"]*100:10.2f}')
print(f'{"Fine-tuned (best_s3)":20s}  {agg_ft["max_err_s"]:10.2e}  {agg_ft["mae_s"]:10.2e}  '
      f'{agg_ft["mean_err_s"]:10.2e}  {agg_ft["var_s2"]:10.2e}  '
      f'{agg_ft["precision"]:10.4f}  {agg_ft["recall"]:10.4f}  {agg_ft["f1"]*100:10.2f}')


  in vivo  —  No Fine-tuning  vs  Fine-tuned (best_s3)  (256 sources)
  Errors: prediction - manual
Model                    Max Err         MAE        Bias   Var (s^2)   Precision      Recall       F1(%)
--------------------------------------------------------------------------------------------------------------
No fine-tuning          6.55e-06    2.61e-06   -2.47e-06    6.55e-12      0.9562      0.9985       97.66
Fine-tuned (best_s3)    2.44e-06    4.61e-07   -7.62e-08    2.82e-12      0.9963      0.9963       99.60
